# 02 — Data preparation

Prepare the lecturer-provided mushroom dataset while keeping the test set isolated. Poisonous mushrooms are encoded as the positive class (1). The synthetic noise columns can be included or excluded through one configuration switch.

Use the saved raw train/test splits for model comparison and stratified cross-validation. Put a fresh preprocessor from build_preprocessor(...) inside a scikit-learn Pipeline so each cross-validation fold learns preprocessing only from its fold's training rows. The separately saved transformed files are for inspection or final-model workflows, not cross-validation on a pre-transformed matrix.

No classifier is trained in this notebook.

## 1. Imports and configuration

Centralize the input path, feature schema, target mapping, split seed, and noise-feature switch. The expected schema is specific to the lecturer-modified dataset.

In [31]:
# Standard-library helpers for paths, JSON metadata, and Python import resolution.
from pathlib import Path
import json
import math
import sys

# Third-party tools used to inspect, transform, and persist the prepared data.
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
# Keep the same deterministic stratified split strategy for later model evaluation.
from sklearn.model_selection import StratifiedKFold, train_test_split

# Find the repository root from either the root or a nested notebook working directory.
# This avoids relying on one specific current-working-directory setting in Jupyter.
PROJECT_ROOT = next(
    (
        candidate for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "mushroom" / "preprocessing.py").is_file()
        and (candidate / "requirements.txt").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root containing mushroom/preprocessing.py.")
# Add the project root only if it is not already importable in this kernel.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# Reuse the shared preprocessor rather than copying its transformation logic here.
from mushroom.preprocessing import MISSING_CATEGORY, build_preprocessor

# Configure the input file and target label interpretation used by this project.
DATA_PATH = Path("~/Downloads/mushroom_project_dataset.csv").expanduser()
TARGET = "class"
TARGET_MAP = {"e": 0, "p": 1}  # edible=0, poisonous=1 (positive class)
# These optional synthetic noise predictors can be excluded before the split.
NOISE_COLUMNS = ["jumbled_noise_0", "jumbled_noise_1"]
DROP_NOISE_FEATURES = True
# Reserve one fifth of the rows as a held-out test partition.
TEST_SIZE = 0.20
# Fixed seed makes the train/test split and CV fold definitions reproducible.
RANDOM_STATE = 42

# Expected schema for the lecturer-provided version of the mushroom dataset.
# Explicitly checking it helps catch the wrong input file before preparation begins.
EXPECTED_FEATURES = [
    "cap-diameter", "stem-height", "stem-width", "spore-print-color",
    "gill-color", "habitat", "season", "ring-type", "cap-shape",
    "stem-surface", "jumbled_noise_0", "jumbled_noise_1",
]
EXPECTED_COLUMNS = EXPECTED_FEATURES + [TARGET]

## 2. Load and validate dataset

Fail clearly on a missing/empty file, unexpected schema, duplicate column names, or invalid/missing target labels. Exact duplicate rows are counted and removed before splitting; no invalid labels or unexpected columns are silently discarded.

In [32]:
# Check the input path before reading, so a missing local dataset gives an actionable error.
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Update DATA_PATH to the raw CSV location."
    )

# Load the source table without changing the original file on disk.
df = pd.read_csv(DATA_PATH)
# Stop early if the file is empty or column names are ambiguous.
if df.empty:
    raise ValueError(f"Dataset at {DATA_PATH} contains no rows.")
if df.columns.has_duplicates:
    duplicates = df.columns[df.columns.duplicated()].tolist()
    raise ValueError(f"Dataset contains duplicate column names: {duplicates}")

# Compare both sides of the schema: missing fields and unexpected fields are both errors.
missing_columns = [column for column in EXPECTED_COLUMNS if column not in df.columns]
unexpected_columns = [column for column in df.columns if column not in EXPECTED_COLUMNS]
if missing_columns or unexpected_columns:
    raise ValueError(
        "Dataset columns do not match the expected lecturer dataset schema. "
        f"Missing: {missing_columns}; unexpected: {unexpected_columns}."
    )
# Reorder only after confirming the exact schema, giving every run a stable input order.
# .copy() makes this a separate working frame rather than a view into the raw read result.
df = df.loc[:, EXPECTED_COLUMNS].copy()
# Labels must be complete and drawn only from the documented encoding before mapping.
if TARGET not in df.columns:
    raise KeyError(f"Expected target column {TARGET!r}; found {df.columns.tolist()}")
if df[TARGET].isna().any():
    raise ValueError(f"Target column {TARGET!r} contains missing labels.")
unknown_labels = set(df[TARGET].unique()).difference(TARGET_MAP)
if unknown_labels:
    raise ValueError(
        f"Unexpected labels in {TARGET!r}: {sorted(map(repr, unknown_labels))}. "
        f"Expected only {sorted(TARGET_MAP)}."
    )

# Count exact duplicates for transparency, then retain only one copy of each identical row.
duplicate_count = int(df.duplicated().sum())
print(f"Raw dataset shape: {df.shape}")
print(f"Exact duplicate rows removed: {duplicate_count}")
df = df.drop_duplicates().copy()
print(f"Validated dataset shape: {df.shape}")

Raw dataset shape: (5000, 13)
Exact duplicate rows removed: 0
Validated dataset shape: (5000, 13)


## 3. Data quality checks

Review data types, missingness, class balance, duplicate feature records, and feature records that occur with conflicting labels. Duplicate feature rows are reported rather than silently removed.

In [33]:
# Inspect inferred data types to distinguish numerical from categorical predictors.
print("Column data types")
display(df.dtypes.to_frame("dtype"))

# Report both the number and percentage of missing entries in each field.
# Sorting brings the most incomplete columns to the top for easier review.
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100),
}).sort_values(["missing_percent", "missing_count"], ascending=[False, False])
print("Missing values by column")
display(missing_summary)

# Establish the original class balance before any split or transformation.
class_summary = df[TARGET].value_counts().rename_axis(TARGET).to_frame("count")
class_summary["percent"] = class_summary["count"].div(len(df)).mul(100)
print("Raw class distribution")
display(class_summary)

# Duplicate feature rows can still be informative when their class labels disagree.
# keep=False marks every member of each duplicate group, not just later occurrences.
feature_duplicate_mask = df[EXPECTED_FEATURES].duplicated(keep=False)
print("Rows participating in duplicate feature records:", int(feature_duplicate_mask.sum()))
if feature_duplicate_mask.any():
    display(df.loc[feature_duplicate_mask].head(20))
# Group on all predictors and count distinct labels to detect contradictory examples.
# dropna=False treats missing predictor values as part of the feature pattern.
conflicting_features = (
    df.groupby(EXPECTED_FEATURES, dropna=False)[TARGET]
    .nunique(dropna=False)
)
conflicting_features = conflicting_features[conflicting_features > 1]
print("Feature records with conflicting labels:", len(conflicting_features))
if not conflicting_features.empty:
    display(conflicting_features.to_frame("distinct_target_labels"))

Column data types


,dtype
cap-diameter,float64
stem-height,float64
stem-width,float64
spore-print-color,str
gill-color,str
habitat,str
season,str
ring-type,str
cap-shape,str
stem-surface,str


Missing values by column


,missing_count,missing_percent
spore-print-color,4740,94.80
stem-surface,3359,67.18
cap-diameter,2032,40.64
stem-height,1013,20.26
habitat,1004,20.08
gill-color,989,19.78
season,973,19.46
ring-type,577,11.54
cap-shape,406,8.12
jumbled_noise_0,406,8.12


Raw class distribution


,count,percent
class,,
e,3107,62.14
p,1893,37.86


Rows participating in duplicate feature records: 0
Feature records with conflicting labels: 0


## 4. Target encoding and feature selection

Encode edible as 0 and poisonous as 1. Excluding the two synthetic jumbled-noise features is an EDA-supported experiment, not proof that they have no predictive value. Set DROP_NOISE_FEATURES to False to retain them in a later comparison. High-missingness biological features remain included.

In [34]:
# Separate predictors from the class label so target values can never enter preprocessing.
# Work on a copy so normalizing predictors leaves the validated source frame unchanged.
X_all = df.drop(columns=[TARGET]).copy()
# Treat whitespace-only strings as missing instead of as a new categorical level.
X_all = X_all.replace(r"^\s*$", np.nan, regex=True)
# Find numeric fields first; infinity is not a valid value for the numeric imputer.
numeric_all = X_all.select_dtypes(include="number").columns.tolist()
if numeric_all:
    infinite_count = int(np.isinf(X_all[numeric_all].to_numpy(dtype=float)).sum())
    if infinite_count:
        print(f"Treating {infinite_count} infinite numeric values as missing.")
        X_all[numeric_all] = X_all[numeric_all].replace([np.inf, -np.inf], np.nan)

# Apply the noise-column option only to columns that are actually present.
dropped_noise_columns = [
    column for column in NOISE_COLUMNS
    if DROP_NOISE_FEATURES and column in X_all.columns
]
# Create the selected predictor matrix and encode the target with poisonous as class 1.
X = X_all.drop(columns=dropped_noise_columns).copy()
y = df[TARGET].map(TARGET_MAP).astype("int8").rename(TARGET)
# Save the original feature order; later assertions use it to detect accidental reordering.
raw_feature_order = X.columns.tolist()
# These column lists are passed into the shared type-specific preprocessing pipeline.
numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = [column for column in X.columns if column not in numeric_cols]

# Derive the expected selected schema independently, then compare it with the actual matrix.
expected_selected_features = [
    column for column in EXPECTED_FEATURES if column not in dropped_noise_columns
]
if raw_feature_order != expected_selected_features:
    raise ValueError(
        f"Selected feature schema changed unexpectedly: {raw_feature_order}; "
        f"expected {expected_selected_features}."
    )
# Prevent downstream estimators from receiving no predictors or an invalid target mapping.
if X.shape[1] == 0:
    raise ValueError("No predictor features remain after target/noise selection.")
if y.isna().any() or set(y.unique()) != {0, 1}:
    raise ValueError("Target encoding did not produce exactly labels 0 (edible) and 1 (poisonous).")

print("Target mapping (1 is poisonous):", TARGET_MAP)
print("Excluded noise columns:", dropped_noise_columns or "none")
print("Raw feature order:", raw_feature_order)
print("Numerical features:", numeric_cols)
print("Categorical features:", categorical_cols)
print("Encoded target counts (0=edible, 1=poisonous):")
display(y.value_counts().sort_index())

Target mapping (1 is poisonous): {'e': 0, 'p': 1}
Excluded noise columns: ['jumbled_noise_0', 'jumbled_noise_1']
Raw feature order: ['cap-diameter', 'stem-height', 'stem-width', 'spore-print-color', 'gill-color', 'habitat', 'season', 'ring-type', 'cap-shape', 'stem-surface']
Numerical features: ['cap-diameter', 'stem-height', 'stem-width']
Categorical features: ['spore-print-color', 'gill-color', 'habitat', 'season', 'ring-type', 'cap-shape', 'stem-surface']
Encoded target counts (0=edible, 1=poisonous):


class
0    3107
1    1893
Name: count, dtype: int64

## 5. Stratified train/test split

Reserve 20% as a final holdout and stratify by the encoded target. Keep the original CSV row indices so partition overlap and feature/label alignment can be checked and preserved in Parquet.

In [35]:
# Stratify so both partitions preserve approximately the original edible/poisonous ratio.
# The test set is held aside here and should not be used for model choice or tuning.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Define the later CV strategy without fitting or evaluating any model in this notebook.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"Training rows: {len(X_train):,}; testing rows: {len(X_test):,}")
print("Training class distribution (fraction; 0=edible, 1=poisonous):")
display(y_train.value_counts(normalize=True).sort_index())
print("Test class distribution (fraction; 0=edible, 1=poisonous):")
display(y_test.value_counts(normalize=True).sort_index())
print("Cross-validation strategy:", cv)

Training rows: 4,000; testing rows: 1,000
Training class distribution (fraction; 0=edible, 1=poisonous):


class
0    0.6215
1    0.3785
Name: proportion, dtype: float64

Test class distribution (fraction; 0=edible, 1=poisonous):


class
0    0.621
1    0.379
Name: proportion, dtype: float64

Cross-validation strategy: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


## 6–8. Define numerical and categorical preprocessing; build a fresh transformer

Numerical measurements are median-imputed and robust-scaled. Missingness indicators are generated separately so they remain binary 0/1. Categorical columns use a dedicated missing token and one-hot encoding with unknown categories ignored. build_preprocessor returns an unfitted ColumnTransformer each time; place it inside the model Pipeline during cross-validation.

In [36]:
# Use the project-wide preprocessing recipe so downstream notebooks stay consistent.
# This returns an unfitted transformer; its statistics/categories are learned from training rows.
preprocessor = build_preprocessor(numeric_cols, categorical_cols)

## 9. Fit and inspect training-only preprocessing

For demonstration and optional final-model workflows, fit one preprocessor on the complete training partition and transform both partitions with it. Do not use these pre-transformed matrices for cross-validation. For leakage-free cross-validation, use Pipeline([("preprocessor", build_preprocessor(numeric_cols, categorical_cols)), ("model", estimator)]) with X_train and y_train.

In [37]:
# Make a fresh transformer for the current train/test operation.
# Fit only on training predictors: fitting on test rows would leak test-set information.
preprocessor = build_preprocessor(numeric_cols, categorical_cols)
X_train_values = preprocessor.fit_transform(X_train)
# Apply the already-fitted training transformations to test rows without refitting.
X_test_values = preprocessor.transform(X_test)
# Keep readable transformed feature names (including one-hot-expanded categories).
feature_names = preprocessor.get_feature_names_out()

# Wrap arrays as DataFrames so original row indices and output feature names are preserved.
# float32 is sufficient for these model inputs and reduces their memory footprint.
X_train_processed = pd.DataFrame(
    X_train_values, columns=feature_names, index=X_train.index
).astype("float32")
X_test_processed = pd.DataFrame(
    X_test_values, columns=feature_names, index=X_test.index
).astype("float32")
print("Transformed feature shapes:", X_train_processed.shape, X_test_processed.shape)
display(X_train_processed.head())

Transformed feature shapes: (4000, 67) (1000, 67)


,numeric__cap-diameter,numeric__stem-height,numeric__stem-width,numeric_missing__missingindicator_cap-diameter,numeric_missing__missingindicator_stem-height,numeric_missing__missingindicator_stem-width,categorical__spore-print-color___missing__,categorical__spore-print-color_g,categorical__spore-print-color_k,categorical__spore-print-color_n,...,categorical__cap-shape_x,categorical__stem-surface___missing__,categorical__stem-surface_f,categorical__stem-surface_g,categorical__stem-surface_h,categorical__stem-surface_i,categorical__stem-surface_k,categorical__stem-surface_s,categorical__stem-surface_t,categorical__stem-surface_y
1647,0.000000,0.435407,1.211562,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4289,1.605839,0.138756,1.790624,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2830,3.748905,-1.732057,0.254554,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
580,0.000000,-2.870813,-1.106631,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1938,-0.548905,-2.870813,-1.106631,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 10. Validate partitions and transformed data

Check split separation, label semantics, stratification, raw and transformed schemas, finite numeric outputs, and feature/target row alignment. Also exercise missing numeric data and an unseen category through the fitted transformer.

In [38]:
# Ensure no source row was assigned to both partitions.
if not set(X_train.index).isdisjoint(X_test.index):
    raise ValueError("Training and test partitions overlap in original row indices.")
# Recompute the documented target mapping and compare it with the prepared target series.
if not y.equals(df.loc[y.index, TARGET].map(TARGET_MAP).astype("int8")):
    raise ValueError("Encoded labels do not match TARGET_MAP.")
if TARGET_MAP != {"e": 0, "p": 1}:
    raise ValueError("Target mapping must encode poisonous mushrooms as positive class 1.")
# Verify that raw and transformed feature schemas keep a stable, identical column order.
if X_train.columns.tolist() != raw_feature_order or X_test.columns.tolist() != raw_feature_order:
    raise ValueError("Raw train/test feature columns do not match the configured feature order.")
if X_train_processed.columns.tolist() != X_test_processed.columns.tolist():
    raise ValueError("Transformed train/test feature columns or order differ.")
# Estimators need numeric finite values; check both splits after preprocessing.
if not np.issubdtype(X_train_processed.to_numpy().dtype, np.number):
    raise TypeError("Transformed training features are not numerical.")
if not np.isfinite(X_train_processed.to_numpy()).all():
    raise ValueError("Non-finite values remain in transformed training features.")
if not np.isfinite(X_test_processed.to_numpy()).all():
    raise ValueError("Non-finite values remain in transformed test features.")
# Index equality confirms every feature row still matches its original target label.
if not X_train.index.equals(y_train.index) or not X_test.index.equals(y_test.index):
    raise ValueError("Predictor rows and target labels are misaligned.")

# Measure class proportions on the whole set and each partition to verify stratification.
overall_distribution = y.value_counts(normalize=True).sort_index()
train_distribution = y_train.value_counts(normalize=True).sort_index()
test_distribution = y_test.value_counts(normalize=True).sort_index()
# A one-percentage-point tolerance allows rounding while catching an unexpectedly skewed split.
if (train_distribution - overall_distribution).abs().max() > 0.01:
    raise ValueError("Training class distribution differs by more than one percentage point.")
if (test_distribution - overall_distribution).abs().max() > 0.01:
    raise ValueError("Test class distribution differs by more than one percentage point.")
if len(df) == 5000 and (len(X_train), len(X_test)) != (4000, 1000):
    raise ValueError("Expected a 4,000/1,000 split for this 5,000-row dataset.")

# Exercise realistic inference inputs: a missing number and a category absent from training.
# Calling transform (not fit/fit_transform) confirms the learned schema remains fixed.
new_observation = X_train.iloc[[0]].copy()
if numeric_cols:
    new_observation.loc[new_observation.index[0], numeric_cols[0]] = np.nan
if categorical_cols:
    new_observation.loc[new_observation.index[0], categorical_cols[0]] = "__unseen_category__"
new_values = preprocessor.transform(new_observation)
if new_values.shape[1] != X_train_processed.shape[1] or not np.isfinite(new_values).all():
    raise ValueError("Fitted preprocessing failed for a new observation with missing/unseen values.")

print("Validation passed: disjoint splits; poisonous=1; stratification preserved; schema/row alignment stable; transformed values finite and numeric.")

Validation passed: disjoint splits; poisonous=1; stratification preserved; schema/row alignment stable; transformed values finite and numeric.


## 11. Save raw splits and preprocessing artifacts

Save raw predictors and encoded labels with their original row indices, plus optional transformed datasets, the training-fitted preprocessor, and JSON metadata. The raw files are the required inputs for cross-validation; the preprocessor is never fitted using the test partition.

In [39]:
# Locate the repository root so generated artifacts always go to the project data directory.
project_root = next(
    (
        candidate for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "mushroom" / "notebooks" / "02_data_preparation.ipynb").is_file()
        and (candidate / "requirements.txt").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate the repository root from the notebook working directory."
    )
# Create the output directory if this is the first run; existing files remain untouched
# except for artifacts with the exact filenames written below.
output_dir = project_root / "mushroom" / "data"
output_dir.mkdir(parents=True, exist_ok=True)

# Persist raw partitions for leakage-safe pipelines; indices preserve X/y row alignment.
X_train.to_parquet(output_dir / "mushroom_X_train_raw.parquet", index=True)
X_test.to_parquet(output_dir / "mushroom_X_test_raw.parquet", index=True)
y_train.to_frame().to_parquet(output_dir / "mushroom_y_train.parquet", index=True)
y_test.to_frame().to_parquet(output_dir / "mushroom_y_test.parquet", index=True)
# Optional transformed snapshots are for inspection/final-fit workflows, not CV inputs.
X_train_processed.to_parquet(output_dir / "mushroom_X_train.parquet", index=True)
X_test_processed.to_parquet(output_dir / "mushroom_X_test.parquet", index=True)
joblib.dump(preprocessor, output_dir / "mushroom_preprocessor.joblib")

# Record settings and feature schema needed to reproduce and audit data preparation.
metadata = {
    "target_mapping": TARGET_MAP,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "numerical_features": numeric_cols,
    "categorical_features": categorical_cols,
    "dropped_noise_columns": dropped_noise_columns,
    "raw_feature_order": raw_feature_order,
    "source_column_order": df.columns.tolist(),
    "n_training_rows": int(len(X_train)),
    "n_test_rows": int(len(X_test)),
    "scikit_learn_version": sklearn.__version__,
}
# UTF-8 JSON keeps the mapping and feature lists portable across environments.
with (output_dir / "mushroom_preprocessing_metadata.json").open("w", encoding="utf-8") as metadata_file:
    json.dump(metadata, metadata_file, indent=2)

# Reload the saved transformer and confirm serialization preserved its transform behavior.
# Comparing a small test slice is a lightweight check of the persisted inference pipeline.
reloaded_preprocessor = joblib.load(output_dir / "mushroom_preprocessor.joblib")
reloaded_values = reloaded_preprocessor.transform(X_test.iloc[:20])
if not np.allclose(reloaded_values, X_test_values[:20]):
    raise ValueError("Reloaded preprocessor output differs from the in-memory transformer.")
print(f"Raw splits, optional transformed data, transformer, and metadata saved to {output_dir}")

Raw splits, optional transformed data, transformer, and metadata saved to /Users/hamidiqbal/ThomasMore/deep learning/AWS/cloud_AI_challenge/cloud-ai-avenue-5/mushroom/data


## 12. Final data preparation summary

The saved raw files preserve original row indices and use the poisonous-positive target mapping. Keep the holdout untouched until final evaluation; use the raw training split and the provided stratified splitter for all model selection.

In [40]:
# Summarize the prepared row counts and the documented positive-class mapping.
print("Final summary")
print(f"Rows: {len(df):,} | Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Target mapping: edible (e)=0, poisonous (p)=1; train/test positive rates: {train_distribution.loc[1]:.3f}/{test_distribution.loc[1]:.3f}")
print(f"Selected predictors: {len(raw_feature_order)}; dropped noise fields: {dropped_noise_columns or 'none'}")
# Reminder for the next notebook: put a fresh preprocessor inside each model Pipeline
# so every CV fold learns imputation and encoding only from its training fold.
print(f"Cross-validation: 5-fold stratified, shuffled, random_state={RANDOM_STATE}; preprocessor must be inside the model Pipeline.")
print(f"Artifacts: {output_dir}")

Final summary
Rows: 5,000 | Train: 4,000 | Test: 1,000
Target mapping: edible (e)=0, poisonous (p)=1; train/test positive rates: 0.379/0.379
Selected predictors: 10; dropped noise fields: ['jumbled_noise_0', 'jumbled_noise_1']
Cross-validation: 5-fold stratified, shuffled, random_state=42; preprocessor must be inside the model Pipeline.
Artifacts: /Users/hamidiqbal/ThomasMore/deep learning/AWS/cloud_AI_challenge/cloud-ai-avenue-5/mushroom/data
